In [ ]:
# ==============================================================================
# STAGE 3: DIRECT PREFERENCE OPTIMIZATION (DPO ALIGNMENT WORKFLOW)
# ==============================================================================

# ------------------------------------------------------------------------------
# PRE-REQUISITE DEPENDENCY INSTALLATION (One-by-One Approach)
# ------------------------------------------------------------------------------
!pip -q install unsloth
!pip -q install --no-deps trl==0.22.2
!pip -q install -U pymupdf datasets
!pip install peft --no-deps
!pip install trl --no-deps
!pip install accelerator --no-deps
!pip install bitsandbytes --no-deps
!pip install xformers --no-deps

import os
import torch
from datasets import load_dataset
from transformers import TrainingArguments
from unsloth import FastLanguageModel, PatchDPOTrainer
from trl import DPOTrainer

# CRITICAL: Unsloth requires patching the DPOTrainer before importing or running it
PatchDPOTrainer()

# ------------------------------------------------------------------------------
# 1. LOADING THE SFT MODEL
# ------------------------------------------------------------------------------
print("Step 1: Loading the instruction-tuned SFT model and matching adapter layers...")
max_seq_length = 2048

# Load the adapter weights created during Stage 2
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/domain-ai-assistant-finetuning/models/stage2_sft_adapter",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# Safely reuse or wrap the current active PEFT matrix configurations
if hasattr(model, "peft_config") or hasattr(model, "active_adapters"):
    print("✨ Detected active Stage 2 SFT adapters attached to the model layout.")
    try:
        # Scale parameters inside the configuration dictionary natively for contrastive tuning
        for adapter_name, config in model.peft_config.items():
            config.lora_alpha = 32  # Keep aligned with SFT configuration for stability
        print("✅ Active adapter configuration balanced and locked for preference optimization.")
    except Exception:
        print("ℹ️ Reusing current base active adapter weights directly.")
else:
    print("🔄 Initializing fresh PEFT adapters container for preference tuning...")
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 32,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )

# ------------------------------------------------------------------------------
# 2 & 3. LOADING AND FORMATTING THE PREFERENCE DATASET
# ------------------------------------------------------------------------------
print("\nStep 2 & 3: Loading and mapping the preference dataset...")

# Define the exact instruction prompt template matching Stage 2 for formatting symmetry
def format_dpo_samples(example):
    return {
        "prompt": f"You are an expert customer support assistant. Provide clear, accurate, and structured answers.\n\n### Question:\n{example['prompt']}\n\n### Response:\n",
        "chosen": example["chosen"],
        "rejected": example["rejected"]
    }

# Load the preference dataset JSONL
dataset = load_dataset("json", data_files={"train": "/content/drive/MyDrive/domain-ai-assistant-finetuning/data/preference_dataset.jsonl"})
dataset = dataset.map(format_dpo_samples)

# ------------------------------------------------------------------------------
# 4 & 5. CONFIGURING AND RUNNING DPO ALIGNMENT
# ------------------------------------------------------------------------------
print("\nStep 4 & 5: Initializing DPOTrainer and running alignment loop...")

# Import DPOConfig directly from trl to ensure all internal attributes exist
from trl import DPOConfig

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None, # Unsloth optimizes memory by bypassing a separate explicit reference model
    args = DPOConfig( # Explicit configuration passed to 'args' parameter
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        max_steps = 50,
        learning_rate = 5e-6,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "/content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/stage3_dpo",
        report_to = "none"
    ), # <-- Correctly closing the DPOConfig object here
    beta = 0.1, # Implicit language model penalty constraint factor
    train_dataset = dataset["train"],
    tokenizer = tokenizer,
    max_length = max_seq_length,
    max_prompt_length = 512,
)

dpo_trainer.train()

# ------------------------------------------------------------------------------
# 6. SAVING THE DPO-ALIGNED MODEL
# ------------------------------------------------------------------------------
print("\nStep 6: Archiving final DPO-aligned model and tokenizer configurations...")
output_dpo_path = "/content/drive/MyDrive/domain-ai-assistant-finetuning/models/final_dpo_model"
model.save_pretrained(output_dpo_path)
tokenizer.save_pretrained(output_dpo_path)
print(f"💾 Production model successfully written to: '{output_dpo_path}'")

# ------------------------------------------------------------------------------
# 7. TESTING THE MODEL AFTER DPO
# ------------------------------------------------------------------------------
print("\n" + "="*60 + "\nStep 7: Executing Post-DPO Alignment Verification Inference...\n" + "="*60)

# Switch active parameters to fast decoding optimization kernels
FastLanguageModel.for_inference(model)

# Test query tracking a severe shipping delay scenario
test_query = "I need a full refund. My delivery was delayed by a month and missed my event."

eval_prompt = f"""You are an expert customer support assistant. Provide clear, accurate, and structured answers.

### Question:
{test_query}

### Response:
"""

inputs = tokenizer([eval_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200, use_cache=True)
decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

# Isolate the newly aligned production output text
aligned_response = decoded_output.split("### Response:\n")[-1].strip()

print(f"Test Customer Prompt:\n-> {test_query}\n")
print(f"DPO Aligned Assistant Response:\n-> {aligned_response}\n")
print("="*60 + "\nStage 3 Preference Alignment Notebook Execution Complete!")